# Análisis de información comercial mediante Web Scraping

**Estudiante:** Miguel Steven Morales  
**Asignatura:** Programación para Análisis de Datos  
**Institución:** Institución Universitaria Digital de Antioquia  
**Actividad:** S30 - Evidencia de aprendizaje 1  

**Producto analizado:** Apple iPhone 15 de 128 GB  
**Plataforma:** Mercado Libre Colombia  

## 1. Introducción

El presente proyecto tiene como finalidad aplicar técnicas de extracción
de información web mediante Python para recopilar y estructurar datos
comerciales del Apple iPhone 15 de 128 GB.

Se estudia el funcionamiento de BeautifulSoup, Selenium y Scrapy,
considerando sus características técnicas y sus posibilidades de
aplicación en procesos de análisis de datos.

In [1]:
%pip install requests beautifulsoup4 selenium scrapy pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Herramientas utilizadas

El proyecto se desarrolla mediante Python y Jupyter Notebook.

Se utilizan las bibliotecas BeautifulSoup, Selenium y Scrapy
para estudiar diferentes técnicas de extracción de datos.

Adicionalmente, se emplea Pandas para la organización y
el procesamiento de los resultados obtenidos.

In [2]:
# Bibliotecas utilizadas en el proyecto

from datetime import datetime
from pathlib import Path
from time import perf_counter
import subprocess
import sys
import tempfile

import pandas as pd
import requests
import scrapy

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

## 3. Fuente de información

La información analizada procede de una copia HTML de una publicación
del Apple iPhone 15 de 128 GB en Mercado Libre Colombia.

El archivo fue guardado manualmente para analizar su estructura
y recuperar información comercial del producto.

Este procedimiento permite estudiar técnicas de extracción de datos
sin realizar solicitudes automatizadas adicionales a la plataforma.

In [3]:
# Carga del documento HTML

ruta_archivo = Path("iphone15_mercadolibre.html")

if not ruta_archivo.is_file():
    raise FileNotFoundError(
        f"No se encontró el archivo: {ruta_archivo}"
    )

with ruta_archivo.open(
    mode="r",
    encoding="utf-8",
    errors="replace"
) as archivo:
    contenido_html = archivo.read()

# Interpretación del HTML
soup = BeautifulSoup(contenido_html, "html.parser")

## 4. Extracción mediante BeautifulSoup

BeautifulSoup permite interpretar la estructura HTML de una página web
y localizar información mediante etiquetas y selectores CSS.

Para esta práctica se identificaron los elementos correspondientes
al nombre del producto, precio, vendedor y estado.

La información obtenida se organiza en una estructura de datos
que posteriormente podrá procesarse mediante Pandas.

### Identificación de elementos HTML

Antes de implementar la extracción, se realizó una inspección de la estructura HTML de la publicación seleccionada.

Mediante BeautifulSoup se identificaron las etiquetas y los atributos que contienen la información comercial del producto.

Los principales elementos utilizados fueron:

| Información         | Selector o método utilizado                    |
| ------------------- | ---------------------------------------------- |
| Nombre del producto | Etiqueta `h1`                                  |
| Precio              | Atributo `itemprop="price"`                    |
| Vendedor            | Clase `.ui-seller-data-header__title`          |
| Estado              | Clase `.ui-pdp-subtitle`                       |
| Disponibilidad      | Búsqueda del texto visible en elementos HTML   |
| URL                 | Etiqueta `link` con atributo `rel="canonical"` |

Durante la exploración se encontraron múltiples valores monetarios correspondientes a diferentes ofertas y opciones de financiación.

Por esta razón, se utilizó el atributo `itemprop="price"` para recuperar el precio principal identificado en los datos estructurados de la publicación.

También se detectaron varios vendedores asociados a distintas secciones de la página. Mediante la revisión del HTML y la comparación con la información visual se identificó a CELUMOVIL como vendedor de la oferta seleccionada.

Finalmente, para determinar la disponibilidad, se localizaron elementos HTML que contenían el mensaje «¡Última en Stock!».

Este procedimiento permitió seleccionar los elementos necesarios para estructurar la información comercial del producto.


In [4]:
# Extracción estructurada de datos del producto

def obtener_texto(documento, selector):
    """Extrae el texto de un elemento HTML mediante un selector CSS."""

    elemento = documento.select_one(selector)

    if elemento is None:
        return None

    return elemento.get_text(" ", strip=True)


def extraer_producto(documento):
    """Recupera los principales atributos comerciales del producto."""

    # Nombre del producto
    nombre = obtener_texto(documento, "h1")

    # Precio principal de la publicación
    elemento_precio = documento.select_one('[itemprop="price"]')
    precio = None

    if elemento_precio is not None:
        contenido_precio = elemento_precio.get("content")

        if contenido_precio:
            precio = int(contenido_precio)

    # Vendedor de la oferta seleccionada
    vendedor = obtener_texto(
        documento,
        ".ui-seller-data-header__title"
    )

    if vendedor:
        vendedor = vendedor.replace("Vendido por", "").strip()

    # Estado del producto
    subtitulo = obtener_texto(
        documento,
        ".ui-pdp-subtitle"
    )

    estado = subtitulo.split("|")[0].strip() if subtitulo else None

    # Identificar el mensaje de disponibilidad del producto
    disponibilidad = None

    for elemento in documento.find_all(["span", "p"]):
        if elemento.find_parent("script"):
            continue

        texto = elemento.get_text(" ", strip=True)

        if texto == "¡Última en Stock!":
            disponibilidad = texto
            break
    
    # Recuperar la URL original de la publicación
    elemento_url = documento.select_one('link[rel="canonical"]')

    url = (
        elemento_url.get("href")
        if elemento_url is not None
        else None
    )

    # Estructuración de la información

    return {
        "Producto": nombre,
        "Precio_COP": precio,
        "Vendedor": vendedor,
        "Estado": estado,
        "Disponibilidad": disponibilidad,
        "URL": url
    }




# Ejecutar la extracción
producto = extraer_producto(soup)

# Registrar la fecha de procesamiento
producto["Fecha_procesamiento"] = (
    datetime.now().astimezone().isoformat(timespec="seconds")
)

# Estructurar los resultados mediante Pandas
df_producto = pd.DataFrame([producto])

display(df_producto)

,Producto,Precio_COP,Vendedor,Estado,Disponibilidad,URL,Fecha_procesamiento
0,Apple iPhone 15 128 GB Negro,2410891,CELUMOVIL,Nuevo,¡Última en Stock!,https://www.mercadolibre.com.co/apple-iphone-1...,2026-10-26T23:16:14-05:00


### Almacenamiento de los resultados

Los datos obtenidos mediante BeautifulSoup se organizan en un DataFrame
de Pandas y posteriormente se exportan a un archivo CSV.

Este formato permite conservar la información estructurada para
su posterior procesamiento y análisis.

In [5]:
# Exportar los resultados de BeautifulSoup

ruta_salida = Path("resultados_beautifulsoup.csv")

df_producto.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8-sig"
)

print(f"Archivo generado: {ruta_salida}")
print(f"Registros exportados: {len(df_producto)}")

Archivo generado: resultados_beautifulsoup.csv
Registros exportados: 1


## 5. Extracción de información mediante Selenium

Selenium es una herramienta de automatización que permite controlar
navegadores web mediante instrucciones programadas.

A diferencia de BeautifulSoup, Selenium permite interactuar con el
documento HTML a través de un navegador, identificar elementos y
recuperar información utilizando distintos mecanismos de búsqueda.

Para esta implementación se utilizará el archivo HTML de la
publicación del Apple iPhone 15 de 128 GB guardado previamente.

El propósito es evaluar la capacidad de Selenium para identificar
y extraer información comercial de la misma fuente utilizada
durante la implementación de BeautifulSoup.

In [6]:
# Preparación del documento HTML para Selenium


# Preparar una copia local del documento sin scripts ni recursos embebidos
documento_selenium = BeautifulSoup(contenido_html, "html.parser")

for elemento in documento_selenium.find_all(
    ["script", "style", "iframe"]
):
    elemento.decompose()

html_simplificado = str(documento_selenium)

### Extracción de atributos comerciales

Una vez comprobado el funcionamiento de Selenium, se procede
a identificar los atributos comerciales de la publicación mediante
selectores CSS y etiquetas HTML.

Se recuperan el nombre del producto, su precio, el vendedor,
la condición y el mensaje de disponibilidad.

Los resultados se organizan posteriormente mediante Pandas
para facilitar su comparación con BeautifulSoup.

In [7]:
# Función para recuperar los atributos comerciales mediante Selenium


def obtener_texto_selenium(navegador, selector):
    """Obtiene el texto de un elemento identificado mediante CSS."""

    try:
        elemento = navegador.find_element(By.CSS_SELECTOR, selector)
        return elemento.text.strip()

    except NoSuchElementException:
        return None


def extraer_producto_selenium(navegador):
    """Extrae los principales atributos de la publicación."""

    nombre = obtener_texto_selenium(navegador, "h1")

    # Precio estructurado del producto
    precio = None

    try:
        elemento_precio = navegador.find_element(
            By.CSS_SELECTOR,
            '[itemprop="price"]'
        )

        contenido_precio = elemento_precio.get_attribute("content")

        if contenido_precio:
            precio = int(contenido_precio)

    except NoSuchElementException:
        pass

    # Vendedor de la oferta seleccionada
    vendedor = obtener_texto_selenium(
        navegador,
        ".ui-seller-data-header__title"
    )

    if vendedor:
        vendedor = vendedor.replace("Vendido por", "").strip()

    # Condición del producto
    subtitulo = obtener_texto_selenium(
        navegador,
        ".ui-pdp-subtitle"
    )

    estado = subtitulo.split("|")[0].strip() if subtitulo else None

    # Mensaje de disponibilidad
    disponibilidad = None

    elementos = navegador.find_elements(By.TAG_NAME, "span")

    for elemento in elementos:
        if elemento.text.strip() == "¡Última en Stock!":
            disponibilidad = elemento.text.strip()
            break
    
    # Recuperar la URL original desde el documento HTML
    elemento_url = navegador.find_elements(
        By.CSS_SELECTOR,
        'link[rel="canonical"]'
    )

    url = (
        elemento_url[0].get_attribute("href")
        if elemento_url
        else None
    )

    return {
        "Producto": nombre,
        "Precio_COP": precio,
        "Vendedor": vendedor,
        "Estado": estado,
        "Disponibilidad": disponibilidad,
        "URL": url
    }

In [8]:
# Ejecución de la extracción de datos mediante Selenium

# Configuración del navegador Chrome para ejecución automatizada
opciones = webdriver.ChromeOptions()

# Ejecutar Chrome sin interfaz gráfica
opciones.add_argument("--headless=new")

# Evitar la descarga de imágenes durante la navegación
opciones.add_argument("--blink-settings=imagesEnabled=false")

# Esperar a que el documento HTML termine de cargarse
opciones.page_load_strategy = "normal"

navegador = None

try:
    with tempfile.TemporaryDirectory() as directorio:

        ruta_temporal = Path(directorio) / "publicacion_selenium.html"

        ruta_temporal.write_text(
            html_simplificado,
            encoding="utf-8"
        )

        # Iniciar el navegador automatizado
        navegador = webdriver.Chrome(options=opciones)
        navegador.set_page_load_timeout(15)

        inicio = perf_counter()

        # Cargar el documento local
        navegador.get(ruta_temporal.as_uri())

        # Esperar hasta que el título esté disponible
        WebDriverWait(navegador, 15).until(
            lambda driver: driver.find_elements(By.TAG_NAME, "h1")
        )

        # Ejecutar la función de extracción
        resultado_selenium = extraer_producto_selenium(navegador)

        tiempo_selenium = perf_counter() - inicio

finally:
    if navegador is not None:
        navegador.quit()

# Registrar el tiempo de extracción
print(f"Tiempo de extracción: {tiempo_selenium:.2f} segundos")

Tiempo de extracción: 3.88 segundos


### Almacenamiento de los resultados

Los atributos recuperados mediante Selenium se organizaron en un
DataFrame de Pandas para facilitar su análisis.

La extracción se realizó sobre una copia local simplificada del HTML
de Mercado Libre Colombia. Esta preparación permitió evitar que los
recursos externos de la página interfirieran con la carga del documento.

Los resultados se exportan a un archivo CSV para conservar la
información y compararla posteriormente con BeautifulSoup y Scrapy.

El tiempo registrado corresponde a la carga y extracción mediante
Selenium, sin incluir la preparación previa del documento HTML.

In [9]:
# Incorporar información de trazabilidad al resultado de Selenium

resultado_selenium["Fecha_procesamiento"] = (
    datetime.now().astimezone().isoformat(timespec="seconds")
)

# Organizar los resultados
df_selenium = pd.DataFrame([resultado_selenium])

# Exportar los datos
ruta_salida_selenium = Path("resultados_selenium.csv")

df_selenium.to_csv(
    ruta_salida_selenium,
    index=False,
    encoding="utf-8-sig"
)

print(f"Archivo generado: {ruta_salida_selenium}")
print(f"Registros exportados: {len(df_selenium)}")

display(df_selenium)

Archivo generado: resultados_selenium.csv
Registros exportados: 1


,Producto,Precio_COP,Vendedor,Estado,Disponibilidad,URL,Fecha_procesamiento
0,Apple iPhone 15 128 GB Negro,2410891,CELUMOVIL,Nuevo,¡Última en Stock!,https://www.mercadolibre.com.co/apple-iphone-1...,2026-10-26T23:16:23-05:00


## 6. Extracción de información mediante Scrapy

Scrapy es un framework de Python especializado en la extracción
automatizada de información desde documentos y sitios web.

A diferencia de BeautifulSoup, Scrapy proporciona una arquitectura
basada en spiders que permite organizar la extracción de datos
y administrar su procesamiento y almacenamiento.

### Implementación del spider

Para mantener la consistencia metodológica, se utilizará el mismo
documento HTML de la publicación del Apple iPhone 15 de 128 GB
empleado en BeautifulSoup y Selenium.

El spider identificará los atributos comerciales mediante
selectores CSS y organizará los resultados en una estructura
que permita su exportación a un archivo CSV.

Esta implementación utilizará una copia HTML local, por lo que
no realizará nuevas solicitudes HTTP a Mercado Libre.

In [10]:
# Verificar la disponibilidad del documento fuente para Scrapy

ruta_scrapy = Path("iphone15_mercadolibre.html")

if not ruta_scrapy.is_file():
    raise FileNotFoundError(
        f"No se encontró el documento HTML: {ruta_scrapy}"
    )

# Convertir la ruta local a una URI compatible con Scrapy
url_scrapy = ruta_scrapy.resolve().as_uri()

print(f"Documento fuente: {ruta_scrapy.name}")
print(f"Ruta para Scrapy: {url_scrapy}")

Documento fuente: iphone15_mercadolibre.html
Ruta para Scrapy: file:///C:/Users/migue/OneDrive/Documents/IU%20DIGITAL/EVIDENCIA%20WEB%20SCRAPING/EA1_Web_Scraping/iphone15_mercadolibre.html


In [11]:
# Definición del spider para la extracción de información comercial

codigo_spider = '''
from pathlib import Path
import scrapy


class Iphone15Spider(scrapy.Spider):
    """Extrae información comercial del iPhone 15 desde un HTML local."""

    name = "iphone15"

    custom_settings = {
        "ROBOTSTXT_OBEY": True,
        "LOG_LEVEL": "INFO"
    }

    async def start(self):
        """Genera la solicitud inicial para procesar el HTML local."""

        ruta_html = Path("iphone15_mercadolibre.html").resolve()

        yield scrapy.Request(
            url=ruta_html.as_uri(),
            callback=self.parse,
            dont_filter=True
        )

    def parse(self, response):
        """Extrae los atributos comerciales mediante selectores CSS."""

        # Nombre del producto
        nombre = response.css("h1::text").get()

        if nombre:
            nombre = nombre.strip()

        # Precio principal
        precio_texto = response.css(
            '[itemprop="price"]::attr(content)'
        ).get()

        precio = int(precio_texto) if precio_texto else None

        # Vendedor de la oferta seleccionada
        textos_vendedor = response.css(
            ".ui-seller-data-header__title"
        ).xpath(".//text()").getall()

        vendedor = " ".join(
            texto.strip()
            for texto in textos_vendedor
            if texto.strip()
        )

        if vendedor:
            vendedor = vendedor.replace("Vendido por", "").strip()
        else:
            vendedor = None

        # Estado del producto
        subtitulo = response.css(
            ".ui-pdp-subtitle::text"
        ).get()

        estado = (
            subtitulo.split("|")[0].strip()
            if subtitulo
            else None
        )

        # Disponibilidad
        disponibilidad = response.xpath(
            '//span[normalize-space(.)="¡Última en Stock!"]'
            '/text() | '
            '//p[normalize-space(.)="¡Última en Stock!"]/text()'
        ).get()

        if disponibilidad:
            disponibilidad = disponibilidad.strip()

        # URL original de la publicación
        url = response.css(
            'link[rel="canonical"]::attr(href)'
        ).get()

        # Entregar el registro estructurado
        yield {
            "Producto": nombre,
            "Precio_COP": precio,
            "Vendedor": vendedor,
            "Estado": estado,
            "Disponibilidad": disponibilidad,
            "URL": url
        }
'''

In [12]:
# Guardar la implementación del spider en un archivo Python

ruta_spider = Path("spider_iphone15.py")

ruta_spider.write_text(
    codigo_spider,
    encoding="utf-8"
)

print(f"Archivo creado: {ruta_spider.name}")
print(f"Ruta: {ruta_spider.resolve()}")

Archivo creado: spider_iphone15.py
Ruta: C:\Users\migue\OneDrive\Documents\IU DIGITAL\EVIDENCIA WEB SCRAPING\EA1_Web_Scraping\spider_iphone15.py


In [13]:
# Ejecución del spider y exportación de resultados a CSV

ruta_csv_scrapy = Path("resultados_scrapy.csv")

# Ejecutar Scrapy utilizando el entorno actual de Python
comando = [
    sys.executable,
    "-m",
    "scrapy",
    "runspider",
    str(ruta_spider),
    "-O",
    str(ruta_csv_scrapy)
]

resultado_scrapy = subprocess.run(
    comando,
    capture_output=True,
    text=True,
    check=False
)

# Validar la ejecución y el archivo de resultados

if resultado_scrapy.returncode != 0:
    raise RuntimeError(
        "La ejecución de Scrapy finalizó con errores:\n"
        + resultado_scrapy.stderr
    )

if not ruta_csv_scrapy.is_file():
    raise FileNotFoundError(
        f"Scrapy no generó el archivo: {ruta_csv_scrapy}"
    )

if ruta_csv_scrapy.stat().st_size == 0:
    raise ValueError(
        "Scrapy generó un archivo CSV vacío. "
        "No se extrajeron registros."
    )

# Comprobar que el archivo contiene registros
df_validacion_scrapy = pd.read_csv(ruta_csv_scrapy)

if df_validacion_scrapy.empty:
    raise ValueError(
        "Scrapy finalizó sin extraer registros comerciales."
    )

print("Scrapy se ejecutó correctamente.")
print(f"Archivo generado: {ruta_csv_scrapy.name}")
print(f"Registros extraídos: {len(df_validacion_scrapy)}")

Scrapy se ejecutó correctamente.
Archivo generado: resultados_scrapy.csv
Registros extraídos: 1


In [14]:
# Validación inicial de los resultados obtenidos mediante Scrapy

df_scrapy = pd.read_csv(ruta_csv_scrapy)

print("Cantidad de registros:", len(df_scrapy))
print("\nColumnas obtenidas:")
print(df_scrapy.columns.tolist())

print("\nInformación comercial extraída:")
display(df_scrapy)

Cantidad de registros: 1

Columnas obtenidas:
['Producto', 'Precio_COP', 'Vendedor', 'Estado', 'Disponibilidad', 'URL']

Información comercial extraída:


,Producto,Precio_COP,Vendedor,Estado,Disponibilidad,URL
0,Apple iPhone 15 128 GB Negro,2410891,CELUMOVIL,Nuevo,¡Última en Stock!,https://www.mercadolibre.com.co/apple-iphone-1...


## 7. Comparación de los métodos de extracción

Con el propósito de evaluar la consistencia de la información obtenida,
se comparan los resultados generados mediante BeautifulSoup, Selenium
y Scrapy.

La comparación considera seis atributos comerciales del Apple iPhone 15
de 128 GB: nombre del producto, precio, vendedor, estado, disponibilidad
y URL de la publicación.

Los tres métodos utilizan información correspondiente a una misma
publicación de Mercado Libre Colombia, lo que permite identificar
diferencias en la extracción y evaluar la integridad de los registros.

Esta evaluación compara la consistencia de los datos obtenidos y no
constituye una medición de rendimiento sobre un conjunto amplio de
publicaciones.

In [15]:
# Cargar los resultados generados por los tres métodos

archivos_resultados = {
    "BeautifulSoup": Path("resultados_beautifulsoup.csv"),
    "Selenium": Path("resultados_selenium.csv"),
    "Scrapy": Path("resultados_scrapy.csv")
}

resultados_metodos = {}

for metodo, ruta_archivo in archivos_resultados.items():

    if not ruta_archivo.is_file():
        raise FileNotFoundError(
            f"No se encontró el archivo de {metodo}: {ruta_archivo}"
        )

    if ruta_archivo.stat().st_size == 0:
        raise ValueError(
            f"El archivo de {metodo} está vacío."
        )

    resultados_metodos[metodo] = pd.read_csv(ruta_archivo)

    print(
        f"{metodo}: {len(resultados_metodos[metodo])} "
        "registro(s) cargado(s)"
    )

BeautifulSoup: 1 registro(s) cargado(s)
Selenium: 1 registro(s) cargado(s)
Scrapy: 1 registro(s) cargado(s)


### Comparación de los atributos comerciales

Se realiza una comparación campo por campo de la información
obtenida mediante BeautifulSoup, Selenium y Scrapy.

El análisis considera los atributos de identificación del producto,
precio, vendedor, estado, disponibilidad y URL de la publicación.

Para cada atributo se verifica si los tres métodos recuperaron
el mismo valor. Este procedimiento permite identificar posibles
diferencias en los resultados y evaluar la consistencia de la
extracción de información.

In [16]:
# Comparación de los atributos comerciales entre los tres métodos

campos_comerciales = [
    "Producto",
    "Precio_COP",
    "Vendedor",
    "Estado",
    "Disponibilidad",
    "URL"
]

# Verificar que todos los métodos contienen los campos requeridos
for metodo, dataframe in resultados_metodos.items():

    columnas_faltantes = [
        campo
        for campo in campos_comerciales
        if campo not in dataframe.columns
    ]

    if columnas_faltantes:
        raise ValueError(
            f"{metodo} no contiene las columnas: {columnas_faltantes}"
        )

    if len(dataframe) != 1:
        raise ValueError(
            f"Se esperaba un registro en {metodo}, "
            f"pero se encontraron {len(dataframe)}."
        )

# Construir la tabla comparativa
comparacion = []

for campo in campos_comerciales:

    valores = {
        metodo: dataframe.iloc[0][campo]
        for metodo, dataframe in resultados_metodos.items()
    }

    valores_normalizados = [
        str(valor).strip()
        for valor in valores.values()
        if pd.notna(valor)
    ]

    coincidencia = (
        len(valores_normalizados) == len(resultados_metodos)
        and len(set(valores_normalizados)) == 1
    )

    comparacion.append({
        "Atributo": campo,
        "BeautifulSoup": valores["BeautifulSoup"],
        "Selenium": valores["Selenium"],
        "Scrapy": valores["Scrapy"],
        "Coincidencia": "Sí" if coincidencia else "No"
    })

df_comparacion = pd.DataFrame(comparacion)

display(df_comparacion)

,Atributo,BeautifulSoup,Selenium,Scrapy,Coincidencia
0,Producto,Apple iPhone 15 128 GB Negro,Apple iPhone 15 128 GB Negro,Apple iPhone 15 128 GB Negro,Sí
1,Precio_COP,2410891,2410891,2410891,Sí
2,Vendedor,CELUMOVIL,CELUMOVIL,CELUMOVIL,Sí
3,Estado,Nuevo,Nuevo,Nuevo,Sí
4,Disponibilidad,¡Última en Stock!,¡Última en Stock!,¡Última en Stock!,Sí
5,URL,https://www.mercadolibre.com.co/apple-iphone-1...,https://www.mercadolibre.com.co/apple-iphone-1...,https://www.mercadolibre.com.co/apple-iphone-1...,Sí


### Evaluación de la calidad de los datos

Para evaluar la calidad de la información obtenida mediante
BeautifulSoup, Selenium y Scrapy, se calcularon tres indicadores:

- **Completitud:** porcentaje de atributos comerciales que contienen
  información válida en cada método.
- **Consistencia:** porcentaje de atributos cuyos valores coinciden
  entre los tres métodos de extracción.
- **Cantidad de registros:** número de productos obtenidos mediante
  cada herramienta.

La evaluación se realiza sobre seis atributos comerciales de una
única publicación del Apple iPhone 15 de 128 GB.

Por tanto, los indicadores permiten verificar la calidad de los
resultados de este caso de estudio, pero no representan una
evaluación general del rendimiento de las herramientas.

In [17]:
# Cálculo de indicadores de calidad de los datos extraídos

metricas = []

total_campos = len(campos_comerciales)

for metodo, dataframe in resultados_metodos.items():

    # Seleccionar únicamente los atributos comerciales
    datos = dataframe[campos_comerciales].copy()

    # Identificar valores vacíos o nulos
    valores_validos = (
        datos.notna()
        & datos.astype(str).apply(
            lambda columna: columna.str.strip().ne("")
        )
    )

    # Calcular completitud de los atributos
    completitud = (
        valores_validos.to_numpy().sum()
        / datos.size
    ) * 100

    metricas.append({
        "Metodo": metodo,
        "Registros": len(dataframe),
        "Campos_evaluados": total_campos,
        "Completitud_%": round(completitud, 2)
    })

# Crear tabla de métricas
df_metricas = pd.DataFrame(metricas)

# Calcular consistencia entre los tres métodos
consistencia = (
    df_comparacion["Coincidencia"].eq("Sí").sum()
    / total_campos
) * 100

print("Indicadores de calidad por método:")
display(df_metricas)

print(f"Consistencia entre métodos: {consistencia:.2f}%")

Indicadores de calidad por método:


,Metodo,Registros,Campos_evaluados,Completitud_%
0,BeautifulSoup,1,6,100.0
1,Selenium,1,6,100.0
2,Scrapy,1,6,100.0


Consistencia entre métodos: 100.00%


### Interpretación de los resultados

Los resultados obtenidos permiten comprobar que BeautifulSoup,
Selenium y Scrapy recuperaron correctamente los seis atributos
comerciales definidos para el análisis del Apple iPhone 15 de
128 GB en Mercado Libre Colombia.

Cada herramienta generó un registro con información sobre el
nombre del producto, precio, vendedor, estado, disponibilidad
y dirección de la publicación.

La evaluación de calidad presentó los siguientes resultados:

- **Completitud de BeautifulSoup:** 100 %.
- **Completitud de Selenium:** 100 %.
- **Completitud de Scrapy:** 100 %.
- **Consistencia entre los tres métodos:** 100 %.
- **Registros obtenidos:** uno por cada herramienta.

Estos indicadores demuestran que las tres implementaciones
permitieron recuperar los atributos seleccionados sin valores
faltantes y con resultados coincidentes.

Desde la perspectiva metodológica de CRISP-DM, la actividad
desarrolla principalmente las fases de comprensión y preparación
de los datos, así como una evaluación descriptiva de la calidad
de la información recolectada.

Es importante señalar que los resultados corresponden a una
única publicación y a un documento HTML previamente obtenido.
Por tanto, no permiten establecer conclusiones sobre la
variación de precios en el mercado, la escalabilidad de las
herramientas o su rendimiento en condiciones de navegación
en tiempo real.

La principal contribución del ejercicio consiste en demostrar
la implementación y validación de tres métodos de extracción
de información comercial sobre una fuente común.